# LeetCode 490 — The Maze bằng Python

Link đề: https://leetcode.com/problems/the-maze/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Array, Matrix (đồ thị trên lưới, trạng thái "điểm dừng")

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def hasPath(self, maze: List[List[int]], start: List[int], destination: List[int]) -> bool:
        ...
```

Ghi chú: đây là bài **Premium** của LeetCode. Đề trong notebook được viết lại theo hiểu biết về bài, nên hãy đối chiếu lại đề gốc nếu bạn có tài khoản Premium. Các mê cung phụ do notebook tự dựng được ghi rõ là "tự dựng"; ba ví dụ chính trong mục 2 lấy theo đề gốc (nên đối chiếu lại hình minh họa của đề gốc).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: quả bóng **lăn không dừng** theo một hướng cho đến khi **chạm tường**; chỉ khi dừng mới được chọn hướng mới; hỏi bóng có thể **dừng đúng** ở đích hay không (đi **ngang qua** đích thì **không tính**).
2. Dùng được các công cụ Python: danh sách hướng `[(-1,0), (1,0), (0,-1), (0,1)]`, kiểm tra biên `0 <= x < m`, vòng `while` để "lăn", `collections.deque` (hàng đợi), tập `visited`, đệ quy DFS.
3. Nắm ý then chốt: **đồ thị không phải gồm mọi ô, mà gồm các "điểm dừng"**; mỗi hướng từ một điểm dừng cho đúng một điểm dừng mới (cạnh có hướng).
4. Cài đặt 4 cách khác nhau về ý tưởng: **BFS từng bước trên trạng thái (ô, hướng đang lăn) → DFS đệ quy trên điểm dừng → BFS trên điểm dừng → bảng nhảy tính sẵn "điểm dừng kế tiếp" rồi BFS với cạnh `O(1)`**, và so sánh.
5. Tránh các lỗi kinh điển: coi bài như mê cung thường (đi từng ô), tính "đi ngang qua đích" là thành công, quên đánh dấu `visited`, và lỗi biên với chỉ số âm của Python.

## 2. Đề bài & ví dụ

Có một quả bóng trong `maze` gồm các ô trống (`0`) và tường (`1`). Bóng có thể đi qua các ô trống bằng cách **lăn lên, xuống, trái hoặc phải**, nhưng **không dừng lại cho đến khi chạm tường**. Khi bóng dừng, nó có thể chọn hướng tiếp theo.

Cho mê cung `m x n` `maze`, vị trí xuất phát `start = [start_row, start_col]` và đích `destination = [dest_row, dest_col]`, trả về `True` nếu bóng có thể **dừng lại ở đích**, ngược lại `False`.

Có thể coi **biên của mê cung đều là tường** (ra khỏi lưới nghĩa là chạm tường).

**Ví dụ 1:**

```
maze = [[0,0,1,0,0],
        [0,0,0,0,0],
        [0,0,0,1,0],
        [1,1,0,1,1],
        [0,0,0,0,0]]
start = [0, 4], destination = [4, 4]
```

Kết quả `True`. Một đường đi: trái → xuống → trái → xuống → phải → xuống → phải.

**Ví dụ 2:** cùng `maze`, `start = [0, 4]`, `destination = [3, 2]` → `False`. Bóng có thể **đi ngang qua** `[3, 2]` nhưng không thể **dừng** ở đó.

**Ví dụ 3:**

```
maze = [[0,0,0,0,0],
        [1,1,0,0,1],
        [0,0,0,0,0],
        [0,1,0,0,1],
        [0,1,0,0,0]]
start = [4, 3], destination = [0, 1]
```

Kết quả `False`.

**Ràng buộc:** `1 <= m, n <= 100`; `maze[i][j]` là `0` hoặc `1`; `start`, `destination` có độ dài 2; chỉ số nằm trong lưới; bóng và đích đều ở ô trống và **không trùng nhau ban đầu**; mê cung có ít nhất 2 ô trống.

In [1]:
maze1 = [[0, 0, 1, 0, 0],
         [0, 0, 0, 0, 0],
         [0, 0, 0, 1, 0],
         [1, 1, 0, 1, 1],
         [0, 0, 0, 0, 0]]
maze3 = [[0, 0, 0, 0, 0],
         [1, 1, 0, 0, 1],
         [0, 0, 0, 0, 0],
         [0, 1, 0, 0, 1],
         [0, 1, 0, 0, 0]]

vi_du = [(maze1, [0, 4], [4, 4], True),
         (maze1, [0, 4], [3, 2], False),
         (maze3, [4, 3], [0, 1], False)]

def ve(maze, start, dest):
    """In mê cung: S = xuất phát, D = đích, # = tường, . = ô trống."""
    for i, hang in enumerate(maze):
        print(" ".join("S" if [i, j] == start else "D" if [i, j] == dest else "#" if v else "." for j, v in enumerate(hang)))

ve(maze1, [0, 4], [4, 4])

. . # . S
. . . . .
. . . # .
# # . # #
. . . . D


## 3. Kiến thức nền

### 3.1. Lưới 2 chiều, chỉ số `[hàng][cột]` và kiểm tra biên

`maze[i][j]`: `i` là hàng (từ trên xuống), `j` là cột (từ trái sang). Lưới có `m = len(maze)` hàng và `n = len(maze[0])` cột. Một ô `(x, y)` **nằm trong lưới** khi `0 <= x < m` và `0 <= y < n`. Python cho phép viết chuỗi so sánh `0 <= x < m`. **Cẩn thận:** chỉ số âm trong Python **vòng về cuối** (`maze[-1]` là hàng cuối), nên nếu quên kiểm tra `>= 0` ta không bị lỗi mà bị **sai ngầm**.

In [2]:
m, n = len(maze1), len(maze1[0])
print(m, n)                                   # 5 5
x, y = 0, 4
print(0 <= x < m and 0 <= y < n)              # True
x, y = -1, 4
print(0 <= x < m and 0 <= y < n)              # False
print(maze1[-1])                              # hàng CUỐI! (không báo lỗi) -> bẫy

5 5
True
False
[0, 0, 0, 0, 0]


### 3.2. Danh sách hướng và giải nén tuple

Bốn hướng lăn: lên `(-1, 0)`, xuống `(1, 0)`, trái `(0, -1)`, phải `(0, 1)` — mỗi cặp là **độ thay đổi `(dx, dy)`** của hàng và cột. Vòng `for dx, dy in HUONG:` **giải nén** (unpack) từng cặp vào hai biến.

In [3]:
HUONG = [(-1, 0), (1, 0), (0, -1), (0, 1)]
TEN_HUONG = ["lên", "xuống", "trái", "phải"]
for k, (dx, dy) in enumerate(HUONG):
    print(f"hướng {TEN_HUONG[k]:6}: (dx, dy) = ({dx:>2}, {dy:>2}); từ (2, 2) bước 1 -> ({2 + dx}, {2 + dy})")

hướng lên   : (dx, dy) = (-1,  0); từ (2, 2) bước 1 -> (1, 2)
hướng xuống : (dx, dy) = ( 1,  0); từ (2, 2) bước 1 -> (3, 2)
hướng trái  : (dx, dy) = ( 0, -1); từ (2, 2) bước 1 -> (2, 1)
hướng phải  : (dx, dy) = ( 0,  1); từ (2, 2) bước 1 -> (2, 3)


### 3.3. "Lăn" bằng vòng `while`

Khác mê cung thường (đi một bước rồi chọn lại), ở đây bóng **tiếp tục** cùng một hướng cho đến khi **bước kế tiếp** là tường hoặc ra ngoài lưới. Ta mô tả bằng vòng `while` kiểm tra **ô kế tiếp**: nếu trống thì tiến, ngược lại dừng. Hàm `lan(maze, x, y, dx, dy)` trả về **điểm dừng**.

In [4]:
def lan(maze, x, y, dx, dy):
    m, n = len(maze), len(maze[0])
    while 0 <= x + dx < m and 0 <= y + dy < n and maze[x + dx][y + dy] == 0:
        x += dx
        y += dy
    return x, y

print(lan(maze1, 0, 4, 0, -1))      # lăn sang trái từ (0,4): (0,3) trống, (0,2) là tường -> dừng ở (0,3)
print(lan(maze1, 0, 4, 1, 0))       # lăn xuống từ (0,4): (1,4),(2,4) trống, (3,4) tường -> dừng ở (2,4)
print(lan(maze1, 0, 4, -1, 0))      # lên: ngay sát biên -> không nhúc nhích: (0,4)

(0, 3)
(2, 4)
(0, 4)


### 3.4. Hàng đợi `deque`, tập `visited` và BFS

**BFS** (duyệt theo chiều rộng) dùng một **hàng đợi**: lấy ra từ **đầu** (`popleft`), thêm vào **cuối** (`append`). `collections.deque` làm việc đó trong `O(1)` (khác `list.pop(0)` tốn `O(n)`). Một tập `visited` ghi những nút đã thêm vào hàng đợi để **không xử lý lặp**.

In [5]:
from collections import deque

q = deque([1])
visited = {1}
while q:
    u = q.popleft()
    print("lấy ra", u, end=" -> thêm: ")
    for v in (2 * u, 2 * u + 1):
        if v <= 6 and v not in visited:
            visited.add(v)
            q.append(v)
            print(v, end=" ")
    print()

lấy ra 1 -> thêm: 2 3 
lấy ra 2 -> thêm: 4 5 
lấy ra 3 -> thêm: 6 
lấy ra 4 -> thêm: 
lấy ra 5 -> thêm: 
lấy ra 6 -> thêm: 


### 3.5. Đệ quy DFS và so sánh danh sách

**DFS** (duyệt theo chiều sâu) đi hết một nhánh rồi mới quay lại, thường viết bằng **đệ quy**. Trong bài này `start` và `destination` là **danh sách** `[i, j]`; so sánh `[i, j] == destination` đúng (so từng phần tử). Với DFS đệ quy, Python giới hạn **độ sâu đệ quy** mặc định (khoảng 1000), điều ta sẽ lưu ý ở mục 7.

In [6]:
import sys
print([2, 3] == [2, 3], (2, 3) == [2, 3])        # True False  (tuple khác list!)
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())

def dem_nguoc(k):
    if k == 0:
        return 0
    return 1 + dem_nguoc(k - 1)

print(dem_nguoc(100))

True False
giới hạn đệ quy mặc định: 1000
100


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy nhìn mê cung ví dụ 1 và xem bóng đi từ `S = (0, 4)`:

```
. . # . S
. . . . .
. . . # .
# # . # #
. . . . D
```

**Từ `(0, 4)`:** lên: bị chặn (biên) → ở nguyên; xuống: lăn `(1,4)`, `(2,4)`, dừng vì `(3,4)` là tường ⇒ `(2, 4)`; trái: lăn `(0,3)` rồi gặp tường `(0,2)` ⇒ `(0, 3)`; phải: biên ⇒ ở nguyên.

**Quan sát then chốt:** mỗi lần bóng **dừng**, ta có tối đa bốn lựa chọn và mỗi lựa chọn dẫn đến **đúng một** điểm dừng mới. Vậy ta có một **đồ thị**: nút = các **điểm dừng**, cạnh = "lăn theo một hướng". Câu hỏi: **từ `S` có đường tới `D` trong đồ thị này không?**

**Lưu ý quan trọng:** đi **ngang qua** một ô không có nghĩa là có thể dừng ở đó. Ví dụ 2: đích `(3, 2)` nằm trên đường lăn dọc cột `2` nhưng bóng không dừng ở đó vì `(4, 2)` bên dưới trống; bóng sẽ lăn xuống `(4, 2)`.

**Ví dụ đường đi của ví dụ 1** (từ đề): trái → xuống → trái → xuống → phải → xuống → phải:
`(0,4) → (0,3) → (1,3) → (1,0) → (2,0) → (2,2) → (4,2) → (4,4)`.

In [7]:
vi_tri = (0, 4)
duong = ["trái", "xuống", "trái", "xuống", "phải", "xuống", "phải"]
chi = {"lên": (-1, 0), "xuống": (1, 0), "trái": (0, -1), "phải": (0, 1)}
print(vi_tri, end="")
for ten in duong:
    dx, dy = chi[ten]
    vi_tri = lan(maze1, vi_tri[0], vi_tri[1], dx, dy)
    print(f" -{ten}-> {vi_tri}", end="")
print()
assert list(vi_tri) == [4, 4]

(0, 4) -trái-> (0, 3) -xuống-> (1, 3) -trái-> (1, 0) -xuống-> (2, 0) -phải-> (2, 2) -xuống-> (4, 2) -phải-> (4, 4)


## 5. Cách 1 — Brute Force / baseline: BFS từng bước trên trạng thái (ô, hướng đang lăn)

**Ý tưởng:** mô phỏng **đúng vật lý** của đề, từng ô một. Trạng thái là `(i, j, d)`:

- `d` thuộc `0..3`: bóng **đang lăn** theo hướng `d` và đang ở ô `(i, j)`;
- `d = 4` (ký hiệu `DUNG`): bóng **đang đứng yên** ở `(i, j)`.

Chuyển trạng thái:

- Đang **đứng yên**: chọn bất kỳ hướng `k`, chuyển sang trạng thái `(i, j, k)` (bắt đầu lăn).
- Đang **lăn** theo `d`: nếu ô kế tiếp trong hướng `d` còn trống thì tiến một ô, **giữ nguyên hướng**; ngược lại bóng **dừng** tại `(i, j)`: trạng thái `(i, j, DUNG)`.

Đáp án `True` nếu có thể tới trạng thái **đứng yên** tại đích. Cách này **không dùng** ý "đồ thị điểm dừng", chỉ dựa trên luật chơi từng bước ⇒ làm **oracle** độc lập rất tốt.

In [8]:
DUNG = 4

def vet_can_tung_buoc(maze, start, destination):
    m, n = len(maze), len(maze[0])
    bat_dau = (start[0], start[1], DUNG)
    thay = {bat_dau}
    hang_doi = deque([bat_dau])
    while hang_doi:
        i, j, d = hang_doi.popleft()
        if d == DUNG:
            if [i, j] == destination:
                return True
            ke_tiep = [(i, j, k) for k in range(4)]          # chọn một hướng để bắt đầu lăn
        else:
            dx, dy = HUONG[d]
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and maze[x][y] == 0:
                ke_tiep = [(x, y, d)]                        # còn lăn được
            else:
                ke_tiep = [(i, j, DUNG)]                     # chạm tường: dừng
        for st in ke_tiep:
            if st not in thay:
                thay.add(st)
                hang_doi.append(st)
    return False

for maze, s, d, kq in vi_du:
    print(s, "->", d, ":", vet_can_tung_buoc(maze, s, d))
    assert vet_can_tung_buoc(maze, s, d) == kq

[0, 4] -> [4, 4] : True
[0, 4] -> [3, 2] : False
[4, 3] -> [0, 1] : False


**Độ phức tạp:** có `m · n · 5` trạng thái, mỗi trạng thái xử lý `O(1)` (hoặc `O(4)` khi đứng yên) ⇒ thời gian `O(m · n)` và bộ nhớ `O(m · n)` với hằng số khá lớn (nhiều trạng thái). Thật ra bản này đã **không tệ về độ phức tạp** — nhưng nó mô phỏng cả những ô "đi ngang qua" mà ta không quan tâm, và hằng số lớn. Hãy so sánh số trạng thái với số điểm dừng thật sự:

In [9]:
def dem_trang_thai(maze, start):
    m, n = len(maze), len(maze[0])
    bat_dau = (start[0], start[1], DUNG)
    thay = {bat_dau}
    hang_doi = deque([bat_dau])
    while hang_doi:
        i, j, d = hang_doi.popleft()
        if d == DUNG:
            ke_tiep = [(i, j, k) for k in range(4)]
        else:
            dx, dy = HUONG[d]
            x, y = i + dx, j + dy
            ke_tiep = [(x, y, d)] if 0 <= x < m and 0 <= y < n and maze[x][y] == 0 else [(i, j, DUNG)]
        for st in ke_tiep:
            if st not in thay:
                thay.add(st); hang_doi.append(st)
    return len(thay), sum(1 for (_, _, d) in thay if d == DUNG)

tong, dung = dem_trang_thai(maze1, [0, 4])
print(f"Ví dụ 1: tổng trạng thái đã thăm = {tong}, trong đó số điểm dừng = {dung}")

Ví dụ 1: tổng trạng thái đã thăm = 85, trong đó số điểm dừng = 15


Ta thấy phần lớn trạng thái là "bóng đang lăn giữa đường", không mang thông tin về quyết định. Chỉ các **điểm dừng** mới là nơi ta chọn hướng ⇒ chỉ cần quản lý chúng.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Nút = điểm dừng.** Mỗi điểm dừng có tối đa 4 cạnh ra (một cạnh cho mỗi hướng): "lăn theo hướng đó tới điểm dừng kế".
2. **Lăn cả đoạn trong một bước:** vòng `while` đi tới khi chạm tường, trả về điểm dừng.
3. **Tìm đường trong đồ thị đó:** DFS hoặc BFS với tập `visited` các điểm dừng (không cần `visited` cho các ô lăn ngang qua).
4. **Đáp án** `True` khi điểm dừng `destination` được thăm.
5. **Tối ưu thêm:** tính sẵn **bảng nhảy** `nxt[d][i][j]` = điểm dừng khi lăn từ `(i, j)` theo hướng `d` (DP một lượt mỗi hướng) ⇒ mỗi cạnh `O(1)`, tổng `O(m·n)`.
6. **Lưu ý:** nếu `destination` chỉ là ô **đi ngang qua** thì không tính.

## 7. Các cách tiếp theo

### Cách 2 — DFS đệ quy trên điểm dừng

**Giải thích từng đoạn:**

- `vis[i][j]` đánh dấu điểm dừng `(i, j)` đã thăm.
- `dfs(i, j)`: nếu đã thăm thì thoát; đánh dấu; nếu `[i, j] == destination` có thể dừng sớm (ta chỉ cần biết đã thăm).
- Với mỗi hướng `(dx, dy)`: dùng `lan(...)` để tìm điểm dừng kế `(x, y)`, rồi `dfs(x, y)`.
- Cuối cùng `return vis[destination[0]][destination[1]]`.

In [10]:
def dfs_diem_dung(maze, start, destination):
    m, n = len(maze), len(maze[0])
    vis = [[False] * n for _ in range(m)]

    def dfs(i, j):
        if vis[i][j]:
            return
        vis[i][j] = True
        if [i, j] == destination:
            return
        for dx, dy in HUONG:
            x, y = lan(maze, i, j, dx, dy)
            dfs(x, y)

    dfs(start[0], start[1])
    return vis[destination[0]][destination[1]]

for maze, s, d, kq in vi_du:
    print(s, "->", d, ":", dfs_diem_dung(maze, s, d))
    assert dfs_diem_dung(maze, s, d) == kq

[0, 4] -> [4, 4] : True
[0, 4] -> [3, 2] : False
[4, 3] -> [0, 1] : False


**Code nộp LeetCode (Cách 2):**

```python
from typing import List

class Solution:
    def hasPath(self, maze: List[List[int]], start: List[int], destination: List[int]) -> bool:
        m, n = len(maze), len(maze[0])
        vis = [[False] * n for _ in range(m)]

        def dfs(i, j):
            if vis[i][j]:
                return
            vis[i][j] = True
            if [i, j] == destination:
                return
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                x, y = i, j
                while 0 <= x + dx < m and 0 <= y + dy < n and maze[x + dx][y + dy] == 0:
                    x += dx
                    y += dy
                dfs(x, y)

        dfs(start[0], start[1])
        return vis[destination[0]][destination[1]]
```

**Độ phức tạp:** mỗi điểm dừng được xử lý một lần, mỗi hướng lăn tốn tối đa `O(m + n)` ⇒ thời gian `O(m · n · (m + n))`; bộ nhớ `O(m · n)` (bảng `vis` và ngăn xếp đệ quy).

**Cảnh báo độ sâu đệ quy:** DFS đệ quy có thể sâu tới số điểm dừng (tối đa `m·n = 10^4`). Python mặc định giới hạn khoảng `1000`, nên với mê cung rất lớn có thể gặp `RecursionError`. Ta đo độ sâu DFS thật sự trên mê cung ngẫu nhiên (bằng một DFS viết **không đệ quy** nhưng đi đúng thứ tự):

In [11]:
import random

def do_sau_dfs(maze, start):
    """Độ sâu lớn nhất của cây DFS (giống đệ quy) trên đồ thị điểm dừng; dùng ngăn xếp tường minh."""
    m, n = len(maze), len(maze[0])
    vis = [[False] * n for _ in range(m)]
    vis[start[0]][start[1]] = True
    ngan_xep = [(start[0], start[1], 0)]            # (ô, chỉ số hướng kế tiếp cần thử)
    sau_nhat = 1
    while ngan_xep:
        i, j, k = ngan_xep.pop()
        if k == 4:
            continue
        ngan_xep.append((i, j, k + 1))
        dx, dy = HUONG[k]
        x, y = lan(maze, i, j, dx, dy)
        if not vis[x][y]:
            vis[x][y] = True
            ngan_xep.append((x, y, 0))
            sau_nhat = max(sau_nhat, len(ngan_xep))
    return sau_nhat

def mecung_ngau_nhien(m, n, p_tuong, rng):
    return [[1 if rng.random() < p_tuong else 0 for _ in range(n)] for _ in range(m)]

rng = random.Random(490)
for p in (0.1, 0.25, 0.4):
    mz = mecung_ngau_nhien(100, 100, p, rng)
    mz[0][0] = 0
    print(f"mê cung 100x100, tỉ lệ tường {p}: độ sâu DFS lớn nhất = {do_sau_dfs(mz, [0, 0])}")

mê cung 100x100, tỉ lệ tường 0.1: độ sâu DFS lớn nhất = 529
mê cung 100x100, tỉ lệ tường 0.25: độ sâu DFS lớn nhất = 700
mê cung 100x100, tỉ lệ tường 0.4: độ sâu DFS lớn nhất = 76


Độ sâu có thể lớn tùy mê cung (trường hợp xấu nhất đến hàng nghìn). Vì thế người ta thường ưu tiên **BFS với hàng đợi** (không đệ quy) khi muốn chắc chắn không bị `RecursionError`.

### Cách 3 — BFS trên điểm dừng (không đệ quy)

**Giải thích từng đoạn:**

- `hang_doi = deque([(si, sj)])` chứa điểm dừng cần xử lý; `vis[si][sj] = True`.
- Lấy `(i, j)` ra; nếu `[i, j] == destination` trả `True`.
- Với mỗi hướng: `(x, y) = lan(...)`; nếu `(x, y)` chưa thăm: đánh dấu, đưa vào hàng đợi.
- Hết hàng đợi mà chưa gặp đích ⇒ `False`.

In [12]:
def bfs_diem_dung(maze, start, destination):
    m, n = len(maze), len(maze[0])
    vis = [[False] * n for _ in range(m)]
    vis[start[0]][start[1]] = True
    hang_doi = deque([(start[0], start[1])])
    while hang_doi:
        i, j = hang_doi.popleft()
        if [i, j] == destination:
            return True
        for dx, dy in HUONG:
            x, y = lan(maze, i, j, dx, dy)
            if not vis[x][y]:
                vis[x][y] = True
                hang_doi.append((x, y))
    return False

for maze, s, d, kq in vi_du:
    print(s, "->", d, ":", bfs_diem_dung(maze, s, d))
    assert bfs_diem_dung(maze, s, d) == kq

[0, 4] -> [4, 4] : True
[0, 4] -> [3, 2] : False
[4, 3] -> [0, 1] : False


**Code nộp LeetCode (Cách 3):**

```python
from collections import deque
from typing import List

class Solution:
    def hasPath(self, maze: List[List[int]], start: List[int], destination: List[int]) -> bool:
        m, n = len(maze), len(maze[0])
        vis = [[False] * n for _ in range(m)]
        vis[start[0]][start[1]] = True
        q = deque([(start[0], start[1])])
        while q:
            i, j = q.popleft()
            if [i, j] == destination:
                return True
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                x, y = i, j
                while 0 <= x + dx < m and 0 <= y + dy < n and maze[x + dx][y + dy] == 0:
                    x += dx
                    y += dy
                if not vis[x][y]:
                    vis[x][y] = True
                    q.append((x, y))
        return False
```

**Độ phức tạp:** thời gian `O(m · n · (m + n))` (mỗi điểm dừng lăn 4 hướng, mỗi hướng tối đa `m + n` ô); bộ nhớ `O(m · n)`.

### Cách 4 — Bảng nhảy tính sẵn `nxt[d][i][j]` + BFS với cạnh `O(1)`

**Ý tưởng khác:** thay vì lăn lại từng hướng ở mỗi điểm dừng (tốn `O(m + n)`), ta **tính sẵn** điểm dừng khi lăn từ **mọi ô** theo **mọi hướng** bằng quy hoạch động: nếu ô kế tiếp theo hướng `d` là tường/ra ngoài thì `nxt[d][i][j] = (i, j)`; ngược lại `nxt[d][i][j] = nxt[d][i+dx][j+dy]` (lăn tiếp từ ô kế tiếp). Muốn dùng đúng giá trị đã tính, ta duyệt các ô theo thứ tự **từ phía "đích của hướng lăn" quay về**:

- hướng lên (`dx = -1`): duyệt hàng `i` **tăng dần** (ô phía trên đã tính);
- hướng xuống (`dx = 1`): duyệt hàng `i` **giảm dần**;
- hướng trái (`dy = -1`): duyệt cột `j` **tăng dần**;
- hướng phải (`dy = 1`): duyệt cột `j` **giảm dần**.

Sau đó BFS chỉ tra bảng: `x, y = nxt[d][i][j]` (cạnh `O(1)`).

In [13]:
def tao_bang_nhay(maze):
    m, n = len(maze), len(maze[0])
    nxt = [[[None] * n for _ in range(m)] for _ in range(4)]
    for d, (dx, dy) in enumerate(HUONG):
        hang = range(m) if dx <= 0 else range(m - 1, -1, -1)
        cot = range(n) if dy <= 0 else range(n - 1, -1, -1)
        for i in hang:
            for j in cot:
                if maze[i][j] == 1:
                    continue
                x, y = i + dx, j + dy
                if 0 <= x < m and 0 <= y < n and maze[x][y] == 0:
                    nxt[d][i][j] = nxt[d][x][y]
                else:
                    nxt[d][i][j] = (i, j)
    return nxt

nxt = tao_bang_nhay(maze1)
print("lăn từ (0,4) theo 4 hướng:", [nxt[d][0][4] for d in range(4)])
# đối chiếu với hàm lăn trực tiếp ở MỌI ô trống, MỌI hướng
for d, (dx, dy) in enumerate(HUONG):
    for i in range(5):
        for j in range(5):
            if maze1[i][j] == 0:
                assert nxt[d][i][j] == lan(maze1, i, j, dx, dy)
print("bảng nhảy khớp hàm lan ở mọi ô, mọi hướng")

lăn từ (0,4) theo 4 hướng: [(0, 4), (2, 4), (0, 3), (0, 4)]
bảng nhảy khớp hàm lan ở mọi ô, mọi hướng


In [14]:
def bfs_bang_nhay(maze, start, destination):
    m, n = len(maze), len(maze[0])
    nxt = tao_bang_nhay(maze)
    vis = [[False] * n for _ in range(m)]
    vis[start[0]][start[1]] = True
    hang_doi = deque([(start[0], start[1])])
    while hang_doi:
        i, j = hang_doi.popleft()
        if [i, j] == destination:
            return True
        for d in range(4):
            x, y = nxt[d][i][j]
            if not vis[x][y]:
                vis[x][y] = True
                hang_doi.append((x, y))
    return False

for maze, s, d, kq in vi_du:
    print(s, "->", d, ":", bfs_bang_nhay(maze, s, d))
    assert bfs_bang_nhay(maze, s, d) == kq

[0, 4] -> [4, 4] : True
[0, 4] -> [3, 2] : False
[4, 3] -> [0, 1] : False


**Code nộp LeetCode (Cách 4):**

```python
from collections import deque
from typing import List

class Solution:
    def hasPath(self, maze: List[List[int]], start: List[int], destination: List[int]) -> bool:
        m, n = len(maze), len(maze[0])
        dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        nxt = [[[None] * n for _ in range(m)] for _ in range(4)]
        for d, (dx, dy) in enumerate(dirs):
            rows = range(m) if dx <= 0 else range(m - 1, -1, -1)
            cols = range(n) if dy <= 0 else range(n - 1, -1, -1)
            for i in rows:
                for j in cols:
                    if maze[i][j] == 1:
                        continue
                    x, y = i + dx, j + dy
                    if 0 <= x < m and 0 <= y < n and maze[x][y] == 0:
                        nxt[d][i][j] = nxt[d][x][y]
                    else:
                        nxt[d][i][j] = (i, j)
        vis = [[False] * n for _ in range(m)]
        vis[start[0]][start[1]] = True
        q = deque([(start[0], start[1])])
        while q:
            i, j = q.popleft()
            if [i, j] == destination:
                return True
            for d in range(4):
                x, y = nxt[d][i][j]
                if not vis[x][y]:
                    vis[x][y] = True
                    q.append((x, y))
        return False
```

**Độ phức tạp:** tạo bảng `O(4 · m · n)`, BFS `O(4 · m · n)` ⇒ thời gian `O(m · n)`; bộ nhớ `O(4 · m · n)`.

## 8. TRACE từng bước

BFS trên điểm dừng (Cách 3) với ví dụ 1: `start = (0, 4)`, `destination = (4, 4)`. Ở mỗi bước ta in điểm dừng được lấy ra, kết quả lăn của bốn hướng, và những điểm dừng mới được thêm.

Bảng minh họa các bước đầu (thứ tự hướng: lên, xuống, trái, phải):

| bước | lấy ra | lăn: lên / xuống / trái / phải | điểm dừng mới thêm |
|---|---|---|---|
| 1 | (0,4) | (0,4) / (2,4) / (0,3) / (0,4) | (2,4), (0,3) |
| 2 | (2,4) | (0,4) / (2,4) / (2,4) / (2,4) | không có |
| 3 | (0,3) | (0,3) / (1,3) / (0,3) / (0,4) | (1,3) |
| ... | tiếp tục | | cho tới khi gặp (4,4) |

(Chạy cell dưới để thấy toàn bộ.)

In [15]:
def trace_bfs(maze, start, destination):
    m, n = len(maze), len(maze[0])
    vis = [[False] * n for _ in range(m)]
    vis[start[0]][start[1]] = True
    hang_doi = deque([(start[0], start[1])])
    buoc = 0
    while hang_doi:
        i, j = hang_doi.popleft()
        buoc += 1
        print(f"bước {buoc}: lấy ra {(i, j)}; hàng đợi còn {list(hang_doi)}")
        if [i, j] == destination:
            print("   -> ĐÃ ĐẾN ĐÍCH, trả True")
            return True
        them = []
        ket_qua = []
        for k, (dx, dy) in enumerate(HUONG):
            x, y = lan(maze, i, j, dx, dy)
            ket_qua.append(f"{TEN_HUONG[k]}->{(x, y)}")
            if not vis[x][y]:
                vis[x][y] = True
                hang_doi.append((x, y))
                them.append((x, y))
        print("   lăn:", ", ".join(ket_qua))
        print("   thêm mới:", them)
    print("hết hàng đợi, trả False")
    return False

assert trace_bfs(maze1, [0, 4], [4, 4]) is True

bước 1: lấy ra (0, 4); hàng đợi còn []
   lăn: lên->(0, 4), xuống->(2, 4), trái->(0, 3), phải->(0, 4)
   thêm mới: [(2, 4), (0, 3)]
bước 2: lấy ra (2, 4); hàng đợi còn [(0, 3)]
   lăn: lên->(0, 4), xuống->(2, 4), trái->(2, 4), phải->(2, 4)
   thêm mới: []
bước 3: lấy ra (0, 3); hàng đợi còn []
   lăn: lên->(0, 3), xuống->(1, 3), trái->(0, 3), phải->(0, 4)
   thêm mới: [(1, 3)]
bước 4: lấy ra (1, 3); hàng đợi còn []
   lăn: lên->(0, 3), xuống->(1, 3), trái->(1, 0), phải->(1, 4)
   thêm mới: [(1, 0), (1, 4)]
bước 5: lấy ra (1, 0); hàng đợi còn [(1, 4)]
   lăn: lên->(0, 0), xuống->(2, 0), trái->(1, 0), phải->(1, 4)
   thêm mới: [(0, 0), (2, 0)]
bước 6: lấy ra (1, 4); hàng đợi còn [(0, 0), (2, 0)]
   lăn: lên->(0, 4), xuống->(2, 4), trái->(1, 0), phải->(1, 4)
   thêm mới: []
bước 7: lấy ra (0, 0); hàng đợi còn [(2, 0)]
   lăn: lên->(0, 0), xuống->(2, 0), trái->(0, 0), phải->(0, 1)
   thêm mới: [(0, 1)]
bước 8: lấy ra (2, 0); hàng đợi còn [(0, 1)]
   lăn: lên->(0, 0), xuống->(2, 0), trái->(

**Trace ví dụ 2** (đích `(3, 2)` chỉ là ô đi ngang qua): BFS đi hết mọi điểm dừng nhưng **không bao giờ** có điểm dừng `(3, 2)`. Hãy xem tập điểm dừng thăm được:

In [16]:
def tap_diem_dung(maze, start):
    m, n = len(maze), len(maze[0])
    vis = {(start[0], start[1])}
    hang_doi = deque([(start[0], start[1])])
    while hang_doi:
        i, j = hang_doi.popleft()
        for dx, dy in HUONG:
            x, y = lan(maze, i, j, dx, dy)
            if (x, y) not in vis:
                vis.add((x, y))
                hang_doi.append((x, y))
    return vis

tap = tap_diem_dung(maze1, [0, 4])
print(sorted(tap))
print("(3, 2) có phải điểm dừng?", (3, 2) in tap)       # False -> ví dụ 2 trả False
print("(4, 4) có phải điểm dừng?", (4, 4) in tap)       # True  -> ví dụ 1 trả True
assert (3, 2) not in tap and (4, 4) in tap

[(0, 0), (0, 1), (0, 3), (0, 4), (1, 0), (1, 2), (1, 3), (1, 4), (2, 0), (2, 1), (2, 2), (2, 4), (4, 0), (4, 2), (4, 4)]
(3, 2) có phải điểm dừng? False
(4, 4) có phải điểm dừng? True


## 9. Code nộp LeetCode

Bản dùng Cách 3 (BFS trên điểm dừng, không đệ quy), đúng `class Solution` và chữ ký:

In [17]:
from collections import deque
from typing import List

class Solution:
    def hasPath(self, maze: List[List[int]], start: List[int], destination: List[int]) -> bool:
        m, n = len(maze), len(maze[0])
        vis = [[False] * n for _ in range(m)]
        vis[start[0]][start[1]] = True
        q = deque([(start[0], start[1])])
        while q:
            i, j = q.popleft()
            if [i, j] == destination:
                return True
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                x, y = i, j
                while 0 <= x + dx < m and 0 <= y + dy < n and maze[x + dx][y + dy] == 0:
                    x += dx
                    y += dy
                if not vis[x][y]:
                    vis[x][y] = True
                    q.append((x, y))
        return False

sol = Solution()
for maze, s, d, kq in vi_du:
    kq_thuc = sol.hasPath(maze, s, d)
    print(s, "->", d, ":", kq_thuc)
    assert kq_thuc == kq

[0, 4] -> [4, 4] : True
[0, 4] -> [3, 2] : False
[4, 3] -> [0, 1] : False


## 10. Vì sao thuật toán đúng

**Mô hình hóa.** Bóng chỉ được **đổi hướng khi dừng**. Giữa hai lần dừng nó lăn theo một hướng cố định; điểm dừng được xác định **duy nhất** bởi (điểm dừng trước, hướng). Vậy mọi hành trình hợp lệ là dãy điểm dừng `p0 = start, p1, p2, ..., pk`, trong đó `p_{t+1} = lan(p_t, hướng nào đó)`. Đó chính xác là một **đường đi** trong đồ thị có hướng `G` có tập nút là các ô trống và cạnh `p → lan(p, d)` (với `d` thuộc 4 hướng).

**Bóng dừng đúng ở đích ⇔ đích là một nút đến được từ `start` trong `G`.** Chiều thuận: nếu bóng dừng ở đích thì đích là `p_k` của một dãy hợp lệ. Chiều ngược: nếu đích là nút đến được thì dãy cạnh tương ứng là một hành trình thật. Đi **ngang qua** đích không tạo ra cạnh nào tới đích nên không làm đích "đến được".

**BFS/DFS đúng** vì chúng duyệt đúng tập nút đến được từ `start` (mỗi nút được đánh dấu khi **đưa vào** hàng đợi/ngăn xếp nên không xử lý lặp). `visited` chỉ cần cho **điểm dừng**; một ô có thể bị đi ngang qua nhiều lần (từ nhiều hướng khác nhau) mà không ảnh hưởng.

**Cách 1** theo từng ô tương đương: trạng thái "đứng yên tại `(i, j)`" đạt được ⇔ `(i, j)` là điểm dừng đến được. **Cách 4** chỉ thay hàm `lan` bằng bảng đã tính; bảng đúng theo quy nạp: nếu ô kế tiếp theo hướng `d` bị chặn thì điểm dừng là chính nó, ngược lại giống điểm dừng của ô kế tiếp (đã tính trước nhờ thứ tự duyệt). Ta đã kiểm chứng bảng với hàm `lan` ở mục 7.

## 11. Phân tích Time / Space Complexity

Gọi `m, n` là kích thước, `V ≤ m·n` là số điểm dừng.

| Cách | Time | Space |
|---|---|---|
| 1. BFS từng bước (ô, hướng) | `O(m·n)` (hằng số 5) | `O(m·n)` |
| 2. DFS đệ quy trên điểm dừng | `O(m·n·(m+n))` | `O(m·n)` + ngăn xếp đệ quy |
| 3. BFS trên điểm dừng | `O(m·n·(m+n))` | `O(m·n)` |
| 4. Bảng nhảy + BFS | `O(m·n)` | `O(4·m·n)` |

Vì sao `O(m·n·(m+n))`: mỗi điểm dừng (tối đa `m·n`) lăn 4 hướng, mỗi lần lăn đi tối đa `max(m, n) ≤ m + n` ô. Với `m = n = 100` đó là `10^4 · 4 · 100 = 4·10^6` — đủ nhanh, nên Cách 3 vẫn nộp được.

In [18]:
import time
rng = random.Random(4900)
mz = mecung_ngau_nhien(100, 100, 0.12, rng)
cac_o_trong = [(i, j) for i in range(100) for j in range(100) if mz[i][j] == 0]
s = list(cac_o_trong[0])
d = list(cac_o_trong[-1])
for ten, f in [("từng bước (C1)", vet_can_tung_buoc), ("BFS điểm dừng (C3)", bfs_diem_dung), ("bảng nhảy (C4)", bfs_bang_nhay)]:
    t0 = time.perf_counter()
    kq = f(mz, s, d)
    print(f"{ten:20}: {kq}, {time.perf_counter() - t0:.4f}s")

từng bước (C1)      : True, 0.0138s
BFS điểm dừng (C3)  : True, 0.0057s
bảng nhảy (C4)      : True, 0.0077s


## 12. Bộ test

Chiến lược: ví dụ của đề; biên (lưới `1 x 2`, `2 x 1`, bóng sát đích, đích chỉ đi ngang qua); mê cung **tự dựng** nhỏ có đáp án tính tay; **đối chiếu oracle (Cách 1)** với hàng nghìn mê cung ngẫu nhiên nhỏ (seed cố định); mê cung lớn `100 x 100` đối chiếu các cách còn lại; đảm bảo cả `True` lẫn `False` đều xuất hiện trong test ngẫu nhiên.

In [19]:
CACH = {
    "vet_can_tung_buoc": vet_can_tung_buoc,
    "dfs_diem_dung": dfs_diem_dung,
    "bfs_diem_dung": bfs_diem_dung,
    "bfs_bang_nhay": bfs_bang_nhay,
    "nop": Solution().hasPath,
}

# 1. Ví dụ đề
for maze, s, d, kq in vi_du:
    for ten, f in CACH.items():
        assert f(maze, s, d) == kq, (ten, s, d)
print("ví dụ đề: OK")

# 2. Biên và mê cung tự dựng
tu_dung = [
    ([[0, 0]], [0, 0], [0, 1], True),                        # 1x2: lăn phải dừng ở (0,1)
    ([[0, 0]], [0, 1], [0, 0], True),
    ([[0], [0]], [0, 0], [1, 0], True),                      # 2x1
    ([[0, 0, 0]], [0, 0], [0, 1], False),                    # 1x3: chỉ dừng ở hai đầu, (0,1) chỉ đi ngang qua
    ([[0, 0, 0]], [0, 0], [0, 2], True),
    ([[0, 1, 0]], [0, 0], [0, 2], False),                    # bị tường chắn
]
for maze, s, d, kq in tu_dung:
    for ten, f in CACH.items():
        assert f(maze, s, d) == kq, (ten, maze, s, d)
print("biên + tự dựng: OK")

ví dụ đề: OK
biên + tự dựng: OK


In [20]:
# Mê cung 3x3 trống: từ tâm (1,1) lăn lên (0,1), xuống (2,1), trái (1,0), phải (1,2); sau đó tới các góc
mz3 = [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
tap3 = tap_diem_dung(mz3, [1, 1])
print(sorted(tap3))
assert (0, 0) in tap3                                   # (1,1) -> lên (0,1) -> trái (0,0)
for ten, f in CACH.items():
    assert f(mz3, [1, 1], [0, 0]) is True, ten
    assert f(mz3, [1, 1], [1, 1]) is True, ten          # chính nó (đã ở đó) — test thêm cho các cách (đề cấm trùng nhưng hàm vẫn nên đúng)
print("3x3 trống: OK")

[(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]
3x3 trống: OK


In [21]:
# 3. Ngẫu nhiên nhỏ, so với oracle Cách 1; đếm True/False
rng = random.Random(20490)
dem_true = dem_false = 0
for lan_thu in range(3000):
    m = rng.randint(1, 6)
    n = rng.randint(1, 6)
    if m * n < 2:
        continue
    p = rng.choice([0.1, 0.25, 0.4])
    mz = [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(m)]
    trong = [(i, j) for i in range(m) for j in range(n) if mz[i][j] == 0]
    if len(trong) < 2:
        continue
    (si, sj), (di, dj) = rng.sample(trong, 2)               # hai ô trống KHÁC nhau
    s, d = [si, sj], [di, dj]
    gt = vet_can_tung_buoc(mz, s, d)
    dem_true += gt
    dem_false += (not gt)
    for ten, f in CACH.items():
        assert f(mz, s, d) == gt, (ten, mz, s, d)
print(f"ngẫu nhiên nhỏ: OK  (True: {dem_true}, False: {dem_false})")
assert dem_true > 100 and dem_false > 100

ngẫu nhiên nhỏ: OK  (True: 1438, False: 1371)


In [22]:
# 4. Lớn: 100x100 (các cách nhanh), và mê cung 25x25 cho DFS đệ quy (độ sâu <= số ô trống <= 625)
rng = random.Random(777)
for _ in range(15):
    p = rng.choice([0.05, 0.15, 0.3])
    mz = mecung_ngau_nhien(100, 100, p, rng)
    trong = [(i, j) for i in range(100) for j in range(100) if mz[i][j] == 0]
    (si, sj), (di, dj) = rng.sample(trong, 2)
    s, d = [si, sj], [di, dj]
    gt = bfs_bang_nhay(mz, s, d)
    assert bfs_diem_dung(mz, s, d) == gt
    assert Solution().hasPath(mz, s, d) == gt
    assert vet_can_tung_buoc(mz, s, d) == gt
print("100x100: OK")

for _ in range(80):
    p = rng.choice([0.1, 0.25, 0.4])
    mz = mecung_ngau_nhien(25, 25, p, rng)
    trong = [(i, j) for i in range(25) for j in range(25) if mz[i][j] == 0]
    if len(trong) < 2:
        continue
    (si, sj), (di, dj) = rng.sample(trong, 2)
    s, d = [si, sj], [di, dj]
    assert dfs_diem_dung(mz, s, d) == vet_can_tung_buoc(mz, s, d)
print("DFS đệ quy trên 25x25: OK")

100x100: OK
DFS đệ quy trên 25x25: OK


**Phản ví dụ cho cách "mê cung thường":** nếu coi bài như mê cung đi từng ô (mỗi bước sang ô kề, không cần dừng) thì ví dụ 2 sẽ ra `True` (vì `(3, 2)` là ô trống nối được với `start`), trong khi đáp án đúng là `False`.

In [23]:
def me_cung_thuong(maze, start, destination):
    m, n = len(maze), len(maze[0])
    thay = {tuple(start)}
    q = deque([tuple(start)])
    while q:
        i, j = q.popleft()
        if [i, j] == destination:
            return True
        for dx, dy in HUONG:
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and maze[x][y] == 0 and (x, y) not in thay:
                thay.add((x, y))
                q.append((x, y))
    return False

print("mê cung thường:", me_cung_thuong(maze1, [0, 4], [3, 2]), "| đúng:", Solution().hasPath(maze1, [0, 4], [3, 2]))
assert me_cung_thuong(maze1, [0, 4], [3, 2]) is True and Solution().hasPath(maze1, [0, 4], [3, 2]) is False

mê cung thường: True | đúng: False


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi bài như mê cung thường (đi từng ô một)

Đã thấy ở mục 12. **Vì sao sai:** bóng không được tự do dừng giữa chừng. Đồ thị đúng là đồ thị của **điểm dừng**.

### Lỗi 2 — Kiểm tra đích trong lúc lăn (đi ngang qua cũng tính)

**Code sai:** trong vòng `while` lăn, thấy `(x, y) == destination` là trả `True`.

**Vì sao sai:** đề yêu cầu **dừng** ở đích. Ví dụ 2 cho thấy đi ngang qua không đủ.

In [24]:
def sai_di_ngang_qua(maze, start, destination):
    m, n = len(maze), len(maze[0])
    vis = [[False] * n for _ in range(m)]
    vis[start[0]][start[1]] = True
    q = deque([(start[0], start[1])])
    while q:
        i, j = q.popleft()
        for dx, dy in HUONG:
            x, y = i, j
            while 0 <= x + dx < m and 0 <= y + dy < n and maze[x + dx][y + dy] == 0:
                x += dx
                y += dy
                if [x, y] == destination:                # SAI: mới đi ngang qua
                    return True
            if not vis[x][y]:
                vis[x][y] = True
                q.append((x, y))
    return False

print("sai:", sai_di_ngang_qua(maze1, [0, 4], [3, 2]), "| đúng:", Solution().hasPath(maze1, [0, 4], [3, 2]))
assert sai_di_ngang_qua(maze1, [0, 4], [3, 2]) is True

sai: True | đúng: False


**Code đúng:** chỉ kiểm tra đích khi **lấy ra** một điểm dừng (hoặc xem đích có được thăm).

### Lỗi 3 — Không đánh dấu `visited` (lặp vô hạn / bùng nổ)

Không có `visited` thì bóng có thể lăn qua lại giữa hai điểm dừng mãi mãi. Ta mô phỏng với "cầu chì" đếm số lần gọi để dừng sớm:

In [25]:
class QuaNhieuLanGoi(Exception):
    pass

def dfs_thieu_visited(maze, start, destination, gioi_han=5000):
    dem = [0]

    def dfs(i, j):
        dem[0] += 1
        if dem[0] > gioi_han:
            raise QuaNhieuLanGoi()
        if [i, j] == destination:
            return True
        for dx, dy in HUONG:                          # SAI: không có visited
            x, y = lan(maze, i, j, dx, dy)
            if dfs(x, y):
                return True
        return False

    return dfs(start[0], start[1])

try:
    dfs_thieu_visited(maze1, [0, 4], [3, 2])
except (QuaNhieuLanGoi, RecursionError) as loi:
    print("DFS không có visited bị chặn bởi:", type(loi).__name__, "(lặp vô hạn / tràn đệ quy) trên ví dụ 2")

DFS không có visited bị chặn bởi: RecursionError (lặp vô hạn / tràn đệ quy) trên ví dụ 2


**Code đúng:** đánh dấu điểm dừng khi thăm (DFS) hoặc khi đưa vào hàng đợi (BFS).

### Lỗi 4 — Kiểm tra biên thiếu vế `>= 0` (chỉ số âm vòng về cuối)

**Code sai:** `while x + dx < m and y + dy < n and maze[x + dx][y + dy] == 0:` (quên `0 <=`). Khi lăn lên/trái đến sát biên, `x + dx = -1` mà `maze[-1]` là **hàng cuối** — Python không báo lỗi ngay; bóng bị "dịch chuyển" qua mép bên kia hoặc cuối cùng gây `IndexError`.

In [26]:
def lan_sai_bien(maze, x, y, dx, dy):
    m, n = len(maze), len(maze[0])
    while x + dx < m and y + dy < n and maze[x + dx][y + dy] == 0:     # SAI: thiếu 0 <=
        x += dx
        y += dy
    return x, y

mz = [[0, 0], [0, 0]]                      # lưới 2x2 trống hoàn toàn
print("đúng :", lan(mz, 1, 1, -1, 0))      # lăn lên từ (1,1): dừng ở (0,1)
try:
    print("sai  :", lan_sai_bien(mz, 1, 1, -1, 0))
except IndexError as e:
    print("sai  : IndexError ->", e)

đúng : (0, 1)
sai  : IndexError -> list index out of range


**Code đúng:** `0 <= x + dx < m and 0 <= y + dy < n`.

**Mẹo tránh lỗi:** luôn thử hai ví dụ nhỏ `[[0, 0]]` và `[[0], [0]]` (lưới một hàng / một cột) vì chúng chạm biên ở mọi hướng; và thử ví dụ 2 của đề để chắc rằng "đi ngang qua" không bị tính.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "bóng/đối tượng **lăn tới khi chạm tường**", "trượt đến khi dừng" ⇒ đồ thị trên **điểm dừng**, không phải trên từng ô.
- **Cùng họ:** The Maze II (505, đường ngắn nhất ⇒ Dijkstra vì mỗi cạnh có độ dài khác nhau), The Maze III (499, có lỗ ở giữa đường), Sliding Puzzle, băng trơn (ice sliding).
- **Hai mô hình BFS:** BFS trên điểm dừng (cạnh = cả đoạn lăn) hoặc BFS trên (ô, hướng) (cạnh = một bước). Cái sau luôn đúng cho cả bài có "trọng số theo bước" vì mỗi bước tốn 1.
- **Chọn DFS hay BFS:** nếu chỉ hỏi "có đến được không" thì cả hai đều đúng; để tránh `RecursionError`, chọn BFS hoặc DFS với ngăn xếp tường minh.
- **Tối ưu lăn:** nếu bị TLE, tính sẵn bảng nhảy `nxt[d][i][j]` bằng DP.
- **Luôn kiểm tra biên đủ hai vế** `0 <= x < m`.

## 15. Biến thể

1. **The Maze II (505):** đường **ngắn nhất** (tính số ô đi qua) ⇒ Dijkstra trên điểm dừng, hoặc BFS trên trạng thái từng bước (Cách 1) nhờ mỗi bước tốn 1.
2. **Đếm số điểm dừng** có thể đến được.
3. **Trả về một đường đi** (chuỗi hướng) tới đích: ghi lại cha khi BFS.
4. **The Maze III (499):** thêm một cái lỗ; bóng rơi vào lỗ ngay khi lăn qua nó; yêu cầu đường ngắn nhất và, nếu hòa, nhỏ nhất theo thứ tự từ điển.
5. **Nhiều đích** hoặc **đích là cả một hàng** — chỉnh điều kiện dừng.

In [27]:
def dem_diem_dung_den_duoc(maze, start):
    return len(tap_diem_dung(maze, start))

def tim_duong(maze, start, destination):
    """Biến thể: trả về chuỗi hướng (list tên hướng) tới đích, hoặc None."""
    m, n = len(maze), len(maze[0])
    cha = {(start[0], start[1]): None}
    q = deque([(start[0], start[1])])
    while q:
        i, j = q.popleft()
        if [i, j] == destination:
            duong = []
            cur = (i, j)
            while cha[cur] is not None:
                truoc, k = cha[cur]
                duong.append(TEN_HUONG[k])
                cur = truoc
            return duong[::-1]
        for k, (dx, dy) in enumerate(HUONG):
            x, y = lan(maze, i, j, dx, dy)
            if (x, y) not in cha:
                cha[(x, y)] = ((i, j), k)
                q.append((x, y))
    return None

def mo_phong(maze, start, duong):
    cur = tuple(start)
    for ten in duong:
        dx, dy = HUONG[TEN_HUONG.index(ten)]
        cur = lan(maze, cur[0], cur[1], dx, dy)
    return list(cur)

duong = tim_duong(maze1, [0, 4], [4, 4])
print(duong)
assert mo_phong(maze1, [0, 4], duong) == [4, 4]
assert tim_duong(maze1, [0, 4], [3, 2]) is None
print("số điểm dừng đến được từ (0,4):", dem_diem_dung_den_duoc(maze1, [0, 4]))

['trái', 'xuống', 'trái', 'xuống', 'phải', 'xuống', 'phải']
số điểm dừng đến được từ (0,4): 15


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS từng bước | trạng thái (ô, hướng đang lăn), mô phỏng vật lý | `O(m·n)` (hằng số lớn) | `O(m·n)` | oracle; cũng tính được đường ngắn nhất theo số ô |
| 2. DFS đệ quy | nút = điểm dừng, đệ quy | `O(m·n·(m+n))` | `O(m·n)` + đệ quy | ngắn gọn; coi chừng `RecursionError` |
| 3. BFS điểm dừng | hàng đợi, không đệ quy | `O(m·n·(m+n))` | `O(m·n)` | cách nộp chuẩn, an toàn |
| 4. Bảng nhảy + BFS | tính sẵn điểm dừng kế tiếp bằng DP | `O(m·n)` | `O(4·m·n)` | khi cần tối ưu thời gian / nhiều truy vấn |

**Khuyên dùng khi thi:** Cách 3.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `lan(maze1, 1, 2, 0, 1)`, `lan(maze1, 4, 0, -1, 0)` và `lan(maze1, 2, 2, 1, 0)`; sau đó chạy để kiểm tra.

In [28]:
du_doan_1 = None
du_doan_2 = None
du_doan_3 = None
print("Thực tế:", lan(maze1, 1, 2, 0, 1), lan(maze1, 4, 0, -1, 0), lan(maze1, 2, 2, 1, 0))

Thực tế: (1, 4) (4, 0) (4, 2)


**Bài 2 (tính tay).** Với `maze1`, `start = [0, 4]`: liệt kê **tập các điểm dừng** đến được (dùng BFS bằng tay, ghi lại mỗi điểm dừng lăn 4 hướng ra đâu). Rồi cho biết đích `[2, 2]` có đến được không.

**Bài 3 (điền TODO).** Hoàn thiện hàm `lan_todo(maze, x, y, dx, dy)` trả về điểm dừng.

In [29]:
def lan_todo(maze, x, y, dx, dy):
    m, n = len(maze), len(maze[0])
    # TODO: while ô kế tiếp (x + dx, y + dy) nằm trong lưới (0 <= ... < m / n) VÀ là ô trống: tiến một ô
    # TODO: return x, y
    return None

kq_bai3 = None       # sau khi viết xong: kq_bai3 = lan_todo(maze1, 0, 4, 1, 0)

**Bài 4 (tự viết lại).** Không nhìn lời giải, tự viết `co_duong(maze, start, destination)` bằng **BFS với `deque`** trên điểm dừng.

In [30]:
def co_duong(maze, start, destination):
    # TODO: tự viết BFS
    pass

**Bài 5 (biến thể / mini challenge — The Maze II).** Viết `duong_ngan_nhat(maze, start, destination)` trả về **số ô** bóng đi qua ít nhất để **dừng** ở đích (mỗi ô bóng bước vào tính 1; điểm xuất phát không tính), hoặc `-1` nếu không thể. Gợi ý ở mục 18.

In [31]:
def duong_ngan_nhat(maze, start, destination):
    # TODO: viết tự do (Dijkstra với heapq trên điểm dừng, hoặc BFS trên trạng thái (ô, hướng))
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):**
  - Bài 1: nhìn ô kế tiếp theo hướng đó: nếu là tường hoặc ngoài lưới thì dừng ngay.
  - Bài 3: điều kiện `while`: `0 <= x + dx < m and 0 <= y + dy < n and maze[x + dx][y + dy] == 0`.
  - Bài 5: mỗi lần lăn đếm số ô đi qua.
- **Mức 2 (vừa):**
  - Bài 2: bắt đầu `(0,4)`; mỗi điểm dừng thử 4 hướng và ghi điểm dừng mới theo thứ tự lên, xuống, trái, phải.
  - Bài 4: `q = deque([start]); vis` ...; lấy `popleft`, kiểm tra đích, lăn 4 hướng, thêm điểm dừng chưa thăm.
  - Bài 5: dùng `heapq` với phần tử `(khoảng_cách, i, j)`; khi lăn, đếm số bước `d` và cạnh có trọng số `d`; `dist[i][j] = min(...)`.
- **Mức 3 (gần đáp án):**
  - Bài 2: tập điểm dừng từ `tap_diem_dung(maze1, [0, 4])` có `(2, 2)`.
  - Bài 4: copy Cách 3.
  - Bài 5: `d = dist[i][j] + so_buoc; if d < dist[x][y]: dist[x][y] = d; heappush(h, (d, x, y))`; trả `dist[di][dj]` nếu hữu hạn, ngược lại `-1`.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án được kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert lan(maze1, 1, 2, 0, 1) == (1, 4)       # từ (1,2) sang phải: (1,3),(1,4) trống, hết lưới
assert lan(maze1, 4, 0, -1, 0) == (4, 0)      # lên từ (4,0): (3,0) là tường -> đứng yên
assert lan(maze1, 2, 2, 1, 0) == (4, 2)       # xuống từ (2,2): (3,2),(4,2) trống, hết lưới
print(lan(maze1, 1, 2, 0, 1), lan(maze1, 4, 0, -1, 0), lan(maze1, 2, 2, 1, 0))

# Bài 2
tap = tap_diem_dung(maze1, [0, 4])
print(sorted(tap))
assert (2, 2) in tap
assert all(maze1[i][j] == 0 for i, j in tap)
assert (3, 2) not in tap

(1, 4) (4, 0) (4, 2)
[(0, 0), (0, 1), (0, 3), (0, 4), (1, 0), (1, 2), (1, 3), (1, 4), (2, 0), (2, 1), (2, 2), (2, 4), (4, 0), (4, 2), (4, 4)]


In [33]:
# Bài 3
def lan_dap_an(maze, x, y, dx, dy):
    m, n = len(maze), len(maze[0])
    while 0 <= x + dx < m and 0 <= y + dy < n and maze[x + dx][y + dy] == 0:
        x += dx
        y += dy
    return x, y

assert lan_dap_an(maze1, 0, 4, 1, 0) == (2, 4)
for i in range(5):
    for j in range(5):
        if maze1[i][j] == 0:
            for dx, dy in HUONG:
                assert lan_dap_an(maze1, i, j, dx, dy) == lan(maze1, i, j, dx, dy)

# Bài 4
def co_duong_dap_an(maze, start, destination):
    m, n = len(maze), len(maze[0])
    vis = {(start[0], start[1])}
    q = deque([(start[0], start[1])])
    while q:
        i, j = q.popleft()
        if [i, j] == destination:
            return True
        for dx, dy in HUONG:
            x, y = lan_dap_an(maze, i, j, dx, dy)
            if (x, y) not in vis:
                vis.add((x, y))
                q.append((x, y))
    return False

for maze, s, d, kq in vi_du:
    assert co_duong_dap_an(maze, s, d) == kq

In [34]:
# Bài 5: The Maze II bằng Dijkstra trên điểm dừng
import heapq

def duong_ngan_nhat_dap_an(maze, start, destination):
    m, n = len(maze), len(maze[0])
    INF = float("inf")
    dist = [[INF] * n for _ in range(m)]
    dist[start[0]][start[1]] = 0
    h = [(0, start[0], start[1])]
    while h:
        d0, i, j = heapq.heappop(h)
        if d0 > dist[i][j]:
            continue
        for dx, dy in HUONG:
            x, y, buoc = i, j, 0
            while 0 <= x + dx < m and 0 <= y + dy < n and maze[x + dx][y + dy] == 0:
                x += dx
                y += dy
                buoc += 1
            if buoc > 0 and d0 + buoc < dist[x][y]:
                dist[x][y] = d0 + buoc
                heapq.heappush(h, (d0 + buoc, x, y))
    kq = dist[destination[0]][destination[1]]
    return -1 if kq == INF else kq

# oracle: BFS từng bước, độ sâu của trạng thái "đứng yên tại đích" (mỗi bước tiến một ô tốn 1)
def oracle_ngan_nhat(maze, start, destination):
    m, n = len(maze), len(maze[0])
    bat_dau = (start[0], start[1], DUNG)
    dist = {bat_dau: 0}
    q = deque([bat_dau])
    while q:
        i, j, d = q.popleft()
        if d == DUNG:
            if [i, j] == destination:
                return dist[(i, j, d)]
            nexts = [((i, j, k), 0) for k in range(4)]
        else:
            dx, dy = HUONG[d]
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and maze[x][y] == 0:
                nexts = [((x, y, d), 1)]
            else:
                nexts = [((i, j, DUNG), 0)]
        for st, w in nexts:
            nd = dist[(i, j, d)] + w
            if st not in dist or nd < dist[st]:
                dist[st] = nd
                if w == 0:
                    q.appendleft(st)          # cạnh 0: xử lý trước (0-1 BFS)
                else:
                    q.append(st)
    return -1

assert duong_ngan_nhat_dap_an(maze1, [0, 4], [4, 4]) == 12
assert duong_ngan_nhat_dap_an(maze1, [0, 4], [3, 2]) == -1
rng = random.Random(55)
for _ in range(600):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    if m * n < 2:
        continue
    mz = [[1 if rng.random() < 0.25 else 0 for _ in range(n)] for _ in range(m)]
    trong = [(i, j) for i in range(m) for j in range(n) if mz[i][j] == 0]
    if len(trong) < 2:
        continue
    (si, sj), (di, dj) = rng.sample(trong, 2)
    assert duong_ngan_nhat_dap_an(mz, [si, sj], [di, dj]) == oracle_ngan_nhat(mz, [si, sj], [di, dj])
print("Tất cả đáp án OK")

Tất cả đáp án OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (BFS trên điểm dừng):**

```python
class Solution:
    def hasPath(self, maze: List[List[int]], start: List[int], destination: List[int]) -> bool:
        m, n = len(maze), len(maze[0])
        vis = [[False] * n for _ in range(m)]
        vis[start[0]][start[1]] = True
        q = deque([(start[0], start[1])])
        while q:
            i, j = q.popleft()
            if [i, j] == destination:          # chỉ kiểm tra khi DỪNG
                return True
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                x, y = i, j
                while 0 <= x + dx < m and 0 <= y + dy < n and maze[x + dx][y + dy] == 0:
                    x += dx                    # lăn tới khi chạm tường
                    y += dy
                if not vis[x][y]:
                    vis[x][y] = True
                    q.append((x, y))
        return False
```

**Checklist trước khi submit:**

- [ ] Nút của đồ thị là **điểm dừng**, không phải mọi ô.
- [ ] Vòng lăn kiểm tra **ô kế tiếp**, đủ hai vế biên `0 <= x + dx < m` và `0 <= y + dy < n`.
- [ ] Đích chỉ được tính khi **dừng** (lấy từ hàng đợi), không khi đi ngang qua.
- [ ] `visited` đánh dấu điểm dừng khi đưa vào hàng đợi.
- [ ] So sánh `[i, j] == destination` (list với list), hoặc đổi `destination` thành tuple.
- [ ] Dùng BFS (hoặc DFS có ngăn xếp tường minh) để tránh `RecursionError`.
- [ ] Đã thử ví dụ 2 (đi ngang qua đích), lưới `1 x 2` và `2 x 1`.
- [ ] Độ phức tạp `O(m·n·(m+n))` thời gian, `O(m·n)` bộ nhớ (hoặc `O(m·n)` thời gian nếu dùng bảng nhảy).